# BERT: Tokenization, Sentiment Analysis, NER, and RAG

## Exercise 1: Tokenization with BERT

In [ ]:
!pip install --quiet transformers torch

In [ ]:
from transformers import AutoTokenizer
import torch

bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print("BERT tokenizer loaded successfully.")

In [ ]:
sample_sentence = "BERT understands context by looking at words from both directions."

In [ ]:
# Tokenize the sentence and view how BERT breaks it down
tokens = bert_tokenizer.tokenize(sample_sentence)

print(f"Original sentence: {sample_sentence}")
print(f"Tokens: {tokens}")

In [ ]:
# Prepare the sentence with special tokens, padding, and truncation for model input
encoded_input = bert_tokenizer(
    sample_sentence,
    padding="max_length",
    truncation=True,
    max_length=20,
    return_tensors="pt"
)

print("Encoded input keys:", encoded_input.keys())
print("\nInput IDs:", encoded_input["input_ids"])
print("\nAttention mask:", encoded_input["attention_mask"])

In [ ]:
# Review the token IDs and tokens, identifying the special tokens BERT adds
decoded_tokens = bert_tokenizer.convert_ids_to_tokens(encoded_input["input_ids"][0])

print("Tokens with special tokens and padding:")
for token, token_id in zip(decoded_tokens, encoded_input["input_ids"][0].tolist()):
    print(f"  {token!r:15s} -> {token_id}")

**Special tokens identified**

- **`[CLS]`**: Added at the very beginning of every input. Its final hidden state is commonly used as an aggregate representation of the entire sentence, especially for classification tasks.
- **`[SEP]`**: Added at the end of a sentence (and between sentence pairs for tasks involving two sentences). It tells the model where one segment ends.
- **`[PAD]`**: Added to fill the sequence up to `max_length` when the actual sentence is shorter. The attention mask marks these positions as 0 so the model ignores them during computation.

## Exercise 2: Sentiment Analysis with BERT Pipeline

In [ ]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

print("Sentiment analysis pipeline created.")

In [ ]:
test_sentence = "This course completely changed how I think about machine learning!"

result = sentiment_pipeline(test_sentence)

print(f"Sentence: {test_sentence}")
print(f"Predicted label: {result[0]['label']}")
print(f"Confidence score: {result[0]['score']:.4f}")

In [ ]:
# Test with a few more examples
more_sentences = [
    "The customer service was terrible and the product broke after one day.",
    "I'm not sure how I feel about this update.",
    "Absolutely loved the experience, would recommend to anyone!"
]

for sentence in more_sentences:
    result = sentiment_pipeline(sentence)[0]
    print(f"'{sentence}'")
    print(f"  -> {result['label']} (score: {result['score']:.4f})\n")

## Exercise 3: Building a Custom Sentiment Analyzer

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F


class BERTSentimentAnalyzer:
    def __init__(self, model_name="distilbert-base-uncased-finetuned-sst-2-english"):
        """
        Initializes the tokenizer and model used for sentiment classification.
        """
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.eval()
        self.labels = self.model.config.id2label

    def preprocess(self, text):
        """
        Cleans and tokenizes input text, preparing tensors for the model.
        """
        text = text.strip()
        encoded = self.tokenizer(
            text,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )
        return encoded

    def predict(self, text):
        """
        Predicts sentiment for the given text and returns the label
        with its confidence score.
        """
        inputs = self.preprocess(text)

        with torch.no_grad():
            outputs = self.model(**inputs)
            logits = outputs.logits
            probs = F.softmax(logits, dim=-1)

        pred_idx = torch.argmax(probs, dim=-1).item()
        confidence = probs[0][pred_idx].item()
        label = self.labels[pred_idx]

        return {"text": text, "label": label, "confidence": round(confidence, 4)}


print("BERTSentimentAnalyzer class defined.")

In [ ]:
analyzer = BERTSentimentAnalyzer()

test_texts = [
    "The new update made the app so much faster and easier to use.",
    "I waited an hour and the order still arrived wrong and cold.",
    "It's an okay product, nothing special but does the job."
]

for text in test_texts:
    result = analyzer.predict(text)
    print(f"Text: {result['text']}")
    print(f"  Label: {result['label']} | Confidence: {result['confidence']}\n")

## Exercise 4: Understanding BERT for Named Entity Recognition (NER)

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification


class BERTNamedEntityRecognizer:
    def __init__(self, model_name="dslim/bert-base-NER"):
        """
        Initializes the tokenizer and token-classification model for NER.
        """
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)
        self.model.eval()
        self.id2label = self.model.config.id2label

    def recognize(self, text):
        """
        Identifies named entities in the input text by mapping token-level
        predictions (B-I-O tagging scheme) back to readable entity labels.
        """
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True)
        tokens = self.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

        with torch.no_grad():
            outputs = self.model(**inputs)
            predictions = torch.argmax(outputs.logits, dim=-1)[0]

        entities = []
        for token, pred_id in zip(tokens, predictions.tolist()):
            label = self.id2label[pred_id]
            if token not in self.tokenizer.all_special_tokens and label != "O":
                entities.append((token, label))

        return entities


print("BERTNamedEntityRecognizer class defined.")

In [ ]:
ner_recognizer = BERTNamedEntityRecognizer()

sample_text = "Sundar Pichai is the CEO of Google, which is headquartered in Mountain View, California."

entities = ner_recognizer.recognize(sample_text)

print(f"Text: {sample_text}\n")
print("Detected entities (token, label):")
for token, label in entities:
    print(f"  {token!r:15s} -> {label}")

**Understanding the B-I-O tagging scheme**

- **B-XXX** marks the *beginning* of an entity of type XXX (e.g., `B-PER` for the first token of a person's name).
- **I-XXX** marks tokens *inside* (continuing) an entity that has already begun (e.g., `I-PER` for the second token of a multi-token name).
- **O** marks tokens that are *outside* any entity (not part of a named entity at all).

Common entity types include `PER` (person), `ORG` (organization), `LOC` (location), and `MISC` (miscellaneous).

## Exercise 5: Comparing BERT and GPT

In [ ]:
import pandas as pd

comparison_table = pd.DataFrame({
    "Aspect": [
        "Architecture",
        "Primary Purpose",
        "Attention Direction",
        "Common Use Cases",
        "Strengths",
        "Weaknesses"
    ],
    "BERT": [
        "Encoder-only (stack of Transformer encoders)",
        "Language understanding (bidirectional context)",
        "Bidirectional — sees both left and right context simultaneously",
        "Text classification, NER, question answering, sentence similarity, sentiment analysis",
        "Strong contextual understanding; excels at tasks requiring full-sentence comprehension",
        "Not designed for free-form text generation; outputs are fixed-length representations, not sequences"
    ],
    "GPT": [
        "Decoder-only (stack of Transformer decoders)",
        "Language generation (autoregressive next-token prediction)",
        "Unidirectional (left-to-right) — only sees previous tokens when predicting the next one",
        "Text generation, chatbots, creative writing, code generation, summarization, translation",
        "Excellent at generating coherent, fluent, contextually relevant text of arbitrary length",
        "Weaker at deep bidirectional understanding tasks since it cannot see future tokens during encoding"
    ]
})

comparison_table

**Reflection on differences and similarities**

Both BERT and GPT are built on the Transformer architecture and rely on the same core self-attention mechanism, but they use it in fundamentally different ways suited to different objectives. BERT's bidirectional encoder design makes it the natural choice whenever a task benefits from understanding a complete piece of text at once — such as classifying a sentence's sentiment or identifying entities — because it can use context from both before and after each word. GPT's unidirectional decoder design, in contrast, is purpose-built for generation, since producing text one token at a time naturally requires only looking at what has already been generated. In modern NLP pipelines, the two are often complementary rather than competing: a system might use BERT-style models to understand or retrieve information, and GPT-style models to generate fluent text based on that retrieved information — which is exactly the pattern explored in Exercise 6 (RAG).

## Exercise 6: Exploring BERT Applications in Retrieval-Augmented Generation (RAG)

**What is Retrieval-Augmented Generation (RAG)?**

RAG is an architecture that combines an information retrieval system with a generative language model. Instead of relying solely on the knowledge a generative model memorized during pretraining (which is fixed and can become outdated), a RAG system first retrieves relevant external documents or passages related to a user's query, then feeds those retrieved passages into the generative model alongside the original query. This allows the model to produce answers grounded in up-to-date or domain-specific information that it was never directly trained on, reducing hallucination and improving factual accuracy.

**BERT's role in the retrieval component**

BERT (or BERT-derived models like Sentence-BERT) is commonly used as the retriever's encoder. Its strong bidirectional understanding of language makes it well-suited for converting text into dense vector representations (embeddings) that capture semantic meaning rather than just surface-level keyword overlap. This allows the retrieval step to find documents that are semantically related to a query, even if they don't share exact words with it.

**How BERT generates embeddings for documents and queries**

Both the documents in the knowledge base and the incoming user query are passed through the same (or a paired) BERT-based encoder. The model produces a fixed-length vector for each piece of text — often derived from the `[CLS]` token's final hidden state or by pooling (e.g., mean-pooling) across all token representations. These vectors are designed so that semantically similar texts end up close together in the embedding space, regardless of differences in exact wording.

**Matching queries with documents using a vector database**

All document embeddings are precomputed and stored in a vector database (e.g., FAISS, Pinecone, Weaviate, Chroma), which is optimized for fast similarity search across potentially millions of vectors. When a new query arrives, it is encoded into a vector using the same BERT-based model, and the vector database performs a nearest-neighbor search (commonly using cosine similarity or dot product) to retrieve the top-k most semantically similar documents.

**Example: BERT and GPT working together in a RAG pipeline**

Consider a customer support chatbot for a software company. A user asks, "How do I reset my two-factor authentication?" A BERT-based encoder converts this question into an embedding and searches the vector database of the company's help documentation for the most relevant articles. The top-matching article (perhaps titled "Recovering Account Access") is retrieved and passed, along with the original question, as context into a GPT-style generative model. GPT then generates a natural, conversational answer that is grounded in the specific, accurate information from that retrieved article — rather than relying purely on what it memorized during pretraining, which might be outdated or generic. This combination lets the system stay accurate and up-to-date by simply updating the documents in the vector database, without needing to retrain the generative model itself.